# IR 0 — Get data: NIST IR, SMILES, and functional-group labels

This is the condensed provenance scraper for the IR workshop example. It is run before the workshop, not during it. Starting from a local list of NIST compound IDs, it retrieves the first available NIST IR spectrum through NistChemPy 2.0, derives a canonical SMILES string from NIST's 2-D molfile, assigns functional-group labels with explicit RDKit SMARTS patterns, standardizes spectra to one 500-point grid, and writes one table.

The output has an `IR` column, a `SMILES` column, a shared `wavenumber` column, metadata, and binary categorical columns for the selected functional groups.


In [ ]:
import sys
from importlib.util import find_spec
from io import StringIO
from pathlib import Path

if find_spec('nistchempy') is None:
    raise ModuleNotFoundError(
        'NISTChemPy is missing from this notebook kernel.\n\n'
        f'Current Python: {sys.executable}\n\n'
        'In the VS Code notebook kernel picker, select the Python executable inside this workshop folder\'s .venv directory. ' 
        'Then open the VS Code terminal at the workshop root and run: uv sync --locked'
    )

import jcamp
import nistchempy as nist
import numpy as np
import pandas as pd
from rdkit import Chem
from tqdm.auto import tqdm

DATA_DIR = Path('data')
OUTPUT_DIR = Path('outputs/nist_acquisition')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ID_FILE = DATA_DIR / 'compounds.parquet'
RAW_CHECKPOINT = OUTPUT_DIR / 'ir_raw_checkpoint.parquet'
OUTPUT_FILE = OUTPUT_DIR / 'IR_SMILES_functional_groups.parquet'
RUN_LIVE_COLLECTION = False
MAX_COMPOUNDS = 10  # Use None only for a deliberate, resumable full crawl.
CHECKPOINT_EVERY = 25
request_config = nist.RequestConfig(delay=3.0, max_attempts=3, kwargs={'timeout': 30.0})

if nist.__version__ != '2.0.0':
    raise RuntimeError(f'Tested with nistchempy 2.0.0; found {nist.__version__}.')
compound_ids = pd.read_parquet(ID_FILE)['data'].dropna().astype(str).drop_duplicates().tolist()
print(f'NistChemPy {nist.__version__}; {len(compound_ids):,} candidate IDs; live={RUN_LIVE_COLLECTION}')


In [ ]:
SMARTS = {
    'Vinylic': '[$([CX3]=[CX3])]', 'Arene': 'c', 'Alcohol': '[#6][OX2H]',
    'Ether': '[OD2]([#6])[#6]', 'Amine': '[NX3;H2,H1;!$(NC=O)]', 'Nitrile': '[NX1]#[CX2]',
    'Aldehyde': '[CX3H1](=O)[#6]', 'Ketone': '[#6][CX3](=O)[#6]',
    'CarboxylAcid': '[OX2H][CX3]=[OX1]', 'Ester': '[#6][CX3](=O)[OX2H0][#6]',
    'Amide': '[NX3][CX3](=[OX1])[#6]', 'AcidChloride': '[CX3](=[OX1])[F,Cl,Br,I]',
}
PATTERNS = {name: Chem.MolFromSmarts(pattern) for name, pattern in SMARTS.items()}

def labels_from_smiles(smiles):
    molecule = Chem.MolFromSmiles(smiles)
    if molecule is None:
        raise ValueError('NIST 2-D molfile produced invalid SMILES.')
    return {name: int(molecule.HasSubstructMatch(pattern)) for name, pattern in PATTERNS.items()}

def fetch_record(compound_id):
    compound = nist.get_compound(compound_id, request_config=request_config)
    if compound is None:
        raise LookupError('No unique NIST record.')
    spectra = compound.get_ir_spectra() or []
    if not spectra:
        raise LookupError('No IR spectrum.')
    parsed = jcamp.jcamp_read(StringIO(spectra[0].jdx_text))
    x, y = np.asarray(parsed.get('x', []), dtype=float), np.asarray(parsed.get('y', []), dtype=float)
    if len(x) < 2 or len(x) != len(y) or not (np.isfinite(x).all() and np.isfinite(y).all()):
        raise ValueError('Invalid JCAMP x/y arrays.')
    molblock = compound.get_mol2D()
    molecule = Chem.MolFromMolBlock(molblock, sanitize=True, removeHs=True)
    if molecule is None:
        raise ValueError('NIST 2-D molfile cannot be parsed.')
    smiles = Chem.MolToSmiles(molecule, canonical=True)
    record = {'NIST_molecule': compound_id, 'CAS_NO': compound.cas_rn,
              'Name': parsed.get('title') or compound.name, 'molform': parsed.get('molform') or compound.formula,
              'SMILES': smiles, 'x_raw': x.tolist(), 'y_raw': y.tolist(),
              'xunits': str(parsed.get('xunits', '')).upper(),
              'yunits': str(parsed.get('yunits', '')).upper()}
    record.update(labels_from_smiles(smiles))
    return record

example = fetch_record('C106365')
print({key: example[key] for key in ['CAS_NO', 'Name', 'SMILES', 'Ester', 'Alcohol']})


In [ ]:
if RUN_LIVE_COLLECTION:
    records = pd.read_parquet(RAW_CHECKPOINT).to_dict('records') if RAW_CHECKPOINT.exists() else []
    completed = {record['NIST_molecule'] for record in records}
    candidates = [identifier for identifier in compound_ids if identifier not in completed]
    if MAX_COMPOUNDS is not None:
        candidates = candidates[:MAX_COMPOUNDS]
    failures = []
    for count, identifier in enumerate(tqdm(candidates, desc='Downloading NIST IR records'), start=1):
        try:
            records.append(fetch_record(identifier))
        except Exception as error:
            failures.append({'NIST_molecule': identifier, 'error': repr(error)})
        if count % CHECKPOINT_EVERY == 0:
            pd.DataFrame(records).to_parquet(RAW_CHECKPOINT, index=False)
    pd.DataFrame(records).to_parquet(RAW_CHECKPOINT, index=False)
    pd.DataFrame(failures).to_csv(OUTPUT_DIR / 'retrieval_failures.csv', index=False)
    print(f'Checkpoint contains {len(records):,} retrieved records; failures this run: {len(failures):,}.')


In [ ]:
def standardize_to_model_table(records, n_points=500):
    frame = pd.DataFrame(records).drop_duplicates('CAS_NO').copy()
    keep = frame.apply(lambda row: len(row.x_raw) == len(row.y_raw) and len(row.x_raw) > 1
                       and row.xunits != 'MICROMETERS' and row.yunits != 'TRANSMITTANCE', axis=1)
    frame = frame[keep]
    spectra = []
    for row in frame.to_dict('records'):
        x, y = np.asarray(row['x_raw'], dtype=float), np.asarray(row['y_raw'], dtype=float)
        order = np.argsort(x); x, y = x[order], y[order]
        x, inverse = np.unique(x, return_inverse=True)
        y = np.bincount(inverse, weights=y) / np.bincount(inverse)
        spectra.append((row, x, y))
    low, high = max(x.min() for _, x, _ in spectra), min(x.max() for _, x, _ in spectra)
    if not low < high:
        raise ValueError('No common wavenumber interval remains.')
    grid, rows = np.linspace(low, high, n_points), []
    for row, x, y in spectra:
        values = np.interp(grid, x, y); span = values.max() - values.min()
        if span > 0:
            output = {key: row[key] for key in ['NIST_molecule', 'CAS_NO', 'Name', 'molform', 'SMILES'] + list(SMARTS)}
            output['wavenumber'] = grid.tolist()
            output['IR'] = ((values - values.min()) / span).tolist()
            rows.append(output)
    return pd.DataFrame(rows)

if RUN_LIVE_COLLECTION:
    model_table = standardize_to_model_table(pd.read_parquet(RAW_CHECKPOINT).to_dict('records'))
    model_table.to_parquet(OUTPUT_FILE, index=False)
    assert model_table.IR.map(len).eq(500).all() and model_table.wavenumber.map(len).eq(500).all()
    print(f'Wrote {len(model_table):,} rows to {OUTPUT_FILE}.')


In [ ]:
# Verify how the packaged course release aligns with this scraper's starting point.
raw_reference = pd.read_parquet(DATA_DIR / 'IR_v09_df_trimmed_500_points.parquet').set_index('NIST_molecule')
label_reference = pd.read_parquet(DATA_DIR / 'IR_v09_df_all_features.parquet').set_index('NIST_molecule')
shared = label_reference.index.intersection(raw_reference.index)
assert len(shared) == len(label_reference)
assert raw_reference.loc[shared, ['CAS_NO', 'Name', 'molform']].equals(label_reference.loc[shared, ['CAS_NO', 'Name', 'molform']])
assert all(np.array_equal(np.asarray(raw_reference.at[key, 'x']), np.asarray(label_reference.at[key, 'x']))
           and np.array_equal(np.asarray(raw_reference.at[key, 'y']), np.asarray(label_reference.at[key, 'y'])) for key in shared)
print(f'Provenance verified: {len(label_reference):,} labeled records exactly match the {len(raw_reference):,}-row standardized source table.')
if OUTPUT_FILE.exists():
    fresh = pd.read_parquet(OUTPUT_FILE)
    required = {'IR', 'wavenumber', 'SMILES'} | set(SMARTS)
    assert required.issubset(fresh.columns)
    print(f'Fresh scraper output has the intended IR, SMILES, and categorical-label columns ({len(fresh):,} rows).')
